# PN24 — nearest-child handover cascade

## TL;DR

The nearest-child cascade exactly recovered every next prime after retaining all factor gates, but the compact
90% criterion failed. On 2,000 deterministic opened anchors, 63.65% closed within three candidate states and
83.85% within three handovers. The median visible path had two handovers, while the median proof crossed 6,336
non-base prime gates. This is partial structural support for the ARA handover representation, not a constant-cost
prime locator.


## Context and methods

The base rung keeps integers surviving gates 2 and 7. For each anchor, the nearest surviving lanes below and above
form the local pair. The upper lane is the first candidate. When a later prime gate divides it, the next upper
survivor becomes the candidate. The path terminates after all gates through the candidate's square root have been
cleared.

The protected 87-bit anchor is not present. The development sample and thresholds are defined in the frozen PN24
protocol.


In [1]:
import csv
import json
from collections import Counter
from pathlib import Path

HERE = Path.cwd()
results = json.loads((HERE / 'PN24_NEAREST_HANDOVER_CASCADE_RESULTS.json').read_text(encoding='utf-8'))
validation = json.loads((HERE / 'PN24_NEAREST_HANDOVER_CASCADE_VALIDATION.json').read_text(encoding='utf-8'))
with (HERE / 'PN24_NEAREST_HANDOVER_CASCADE_ANCHORS.csv').open(encoding='utf-8', newline='') as handle:
    anchors = list(csv.DictReader(handle))
with (HERE / 'PN24_NEAREST_HANDOVER_CASCADE_EVENTS.csv').open(encoding='utf-8', newline='') as handle:
    events = list(csv.DictReader(handle))
print(results['status'])
print('validation:', validation['status'], validation['checks_passed'], '/', validation['checks_total'])
assert validation['status'] == 'PASS'
assert len(anchors) == 2007
assert all(row['final_matches_truth'] == 'True' for row in anchors)


PARTIAL STRUCTURAL SUPPORT
validation: PASS 12 / 12


## Data

Seven previously opened scale anchors are combined with 2,000 deterministic anchors sampled from the opened PN19
interval. The sample contains overlapping next-prime labels and is descriptive rather than an independent-event
sample.


In [2]:
print(results['data'])
sample = [row for row in anchors if row['cohort'] == 'sample']
scale = [row for row in anchors if row['cohort'] == 'scale']
assert len(sample) == 2000 and len(scale) == 7


{'scale_anchors': [100000000, 1000000000, 10000000000, 100000000000, 400000000000, 700000000000, 900000000000], 'sample_interval': [4000000000, 4001000000], 'sample_size': 2000, 'sample_seed': 240722, 'sample_distinct_next_prime_labels': 1930, 'sample_rows_are_independent': False, 'protected_87_bit_anchor_used': False}


## Results — visible handover lineage

In [3]:
summary = results['cascade_sample']
for key in (
    'zero_handover_rate',
    'within_two_candidate_states_rate',
    'within_three_candidate_states_rate',
    'within_three_handover_events_rate',
    'mean_handover_events',
    'median_handover_events',
    'max_handover_events',
):
    print(key, summary[key])
print('distribution', summary['handover_event_distribution'])
assert summary['within_three_candidate_states_rate'] == 0.6365
assert summary['within_three_handover_events_rate'] == 0.8385
assert results['decision']['compact_three_candidate_threshold_passed'] is False


zero_handover_rate 0.116
within_two_candidate_states_rate 0.3575
within_three_candidate_states_rate 0.6365
within_three_handover_events_rate 0.8385
mean_handover_events 2.1305
median_handover_events 2.0
max_handover_events 9
distribution {'0': 232, '1': 483, '2': 558, '3': 404, '4': 213, '5': 75, '6': 27, '7': 4, '8': 3, '9': 1}


## Results — fixed rungs

In [4]:
print('rung | exact rate | mean surviving candidates through prime')
for row in results['fixed_rungs_all_anchors']:
    print(row['rung'], f"{row['exact_rate']:.4f}", f"{row['mean_survivor_candidates_through_prime']:.3f}")


rung | exact rate | mean surviving candidates through prime
odd 0.1036 10.063
mod14 0.1156 8.697
through_3 0.1699 5.955
through_5 0.2038 4.848
through_11 0.2252 4.434
through_13 0.2471 4.094
through_17 0.2611 3.877


## Results — visible events versus hidden gate work

In [5]:
for key in (
    'median_handover_events',
    'median_total_nonbase_gate_crossings',
    'median_silent_gate_crossings',
    'median_initial_to_final_delta_ratio',
):
    print(key, summary[key])
assert summary['median_handover_events'] == 2.0
assert summary['median_total_nonbase_gate_crossings'] == 6336.0
assert summary['median_silent_gate_crossings'] == 6334.0


median_handover_events 2.0
median_total_nonbase_gate_crossings 6336.0
median_silent_gate_crossings 6334.0
median_initial_to_final_delta_ratio 0.1111111111111111


In [6]:
print('anchor | base delta | handover gates | final delta | states')
events_by_anchor = {}
for event in events:
    events_by_anchor.setdefault(int(event['anchor']), []).append(int(event['gate']))
for row in scale:
    anchor = int(row['anchor'])
    print(
        anchor,
        int(row['initial_forward_delta']),
        events_by_anchor.get(anchor, []),
        int(row['final_delta']),
        int(row['candidate_states']),
    )


anchor | base delta | handover gates | final delta | states
100000000 1 [17, 643] 7 3
1000000000 3 [23] 7 2
10000000000 1 [101, 33889] 19 3
100000000000 1 [11] 3 2
400000000000 3 [59, 379, 36943] 19 4
700000000000 1 [41149] 9 2
900000000000 1 [634939] 13 2


## Takeaways

1. The nearest lower/upper pair and each releasing-gate handover are exact, reproducible integer objects.
2. The cascade gives a short visible candidate genealogy: median two handovers, maximum nine in this sample.
3. The first child captured only 11.11% of the final correction at the median, and only 63.65% of anchors closed
   within three candidate states. The frozen 90% compact criterion failed.
4. Thousands of silent gates remain necessary to identify the releasing gates and prove the final candidate.
5. PN24 is therefore an exact incremental wheel/trial-division crosswalk and useful ARA event representation, not
   a new constant-operation prime algorithm.
